问答题
1、
(100分)
1、完成Markdown文档的切块过程，并整理思路，提交切块逻辑流程！
2、对切块后的内容进行向量化，生成稠密向量和稀疏向量！ 

In [ ]:
"""
问答题作答：
  1) Markdown 切块 —— 调本地 md_splitter.process_markdown()，7 步管线：
     Step1 输入 → Step2 标题切分（层级追踪 + 代码围栏保护）→ Step3 无标题兜底
     → Step4 超长章节二次切分（表格原子化）+ 短章节合并 → Step5 组装 content
     → Step6 日志统计 → Step7 备份 chunks.json
  2) 切块内容向量化 —— 调本地 bge_m3_embedding_utils.generate_hybrid_embedding()，
     单次产出稠密向量（1024 维） + 稀疏向量（{token_id: weight}），
     适配 Milvus 混合检索。
"""
import json
import sys
from pathlib import Path

# 让同目录下的 .py 模块可被 import（notebook 跑在不同 cwd 下时也安全）
sys.path.insert(0, str(Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()))

from md_splitter import process_markdown  # noqa: E402
from bge_m3_embedding_utils import generate_hybrid_embedding  # noqa: E402

# ============== 输入：与 01-load 对齐，读 ../01-load/result/sample_new.md ==============
NB_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
MD_PATH = (NB_DIR / "../01-load/result/sample_new.md").resolve()
# 02 自己的 result/，与 01-load 的产物目录解耦
RESULT_DIR = (NB_DIR / "result").resolve()
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[输入] {MD_PATH}")
print(f"[输出目录] {RESULT_DIR}")
md_content = MD_PATH.read_text(encoding="utf-8").strip()

# ============== Step1+7：Markdown 切块 ==============
state = {
    "file_title": MD_PATH.stem,        # sample_new
    "md_content": md_content,
    "file_dir": str(RESULT_DIR),       # 备份 chunks.json 到这里
}
state = process_markdown(state)
chunks = state["chunks"]
print(f"\n[切块完成] 共 {len(chunks)} 个 chunk，已备份到 {RESULT_DIR/'chunks.json'}")

# ============== Step2：向量化（稠密 + 稀疏 + L2 归一化） ==============
contents = [c["content"] for c in chunks]
print(f"\n[向量化] 准备嵌入 {len(contents)} 条文本 ...")
embeddings = generate_hybrid_embedding(contents)
if embeddings is None:
    raise RuntimeError("BGE-M3 嵌入失败，请检查 02-chunk-embedding/bge_m3_embedding_utils.py 中的模型路径")

# 稀疏向量 L2 范数自检（review 第二轮要求：归一化后才能入库 Milvus）
import math
sparse_norms = [math.sqrt(sum(w * w for w in s.values())) for s in embeddings["sparse"]]
print(f"[向量化完成] 稠密维度={len(embeddings['dense'][0])}, "
      f"稀疏非零项 min={min(len(s) for s in embeddings['sparse'])} "
      f"max={max(len(s) for s in embeddings['sparse'])} "
      f"avg={sum(len(s) for s in embeddings['sparse'])//len(embeddings['sparse'])}")
print(f"[稀疏向量 L2 范数] min={min(sparse_norms):.4f} max={max(sparse_norms):.4f}（理论值 1.0）")

# ============== 落盘：每个 chunk 自带 title/content/dense/sparse(dict) ==============
#  sparse 直接用 dict（Milvus SPARSE_FLOAT_VECTOR 直接吃的格式），JSON 也会序列化为 {"id": w}
embeddings_payload = []
for chunk, dense_vec, sparse_dict in zip(chunks, embeddings["dense"], embeddings["sparse"]):
    embeddings_payload.append({
        "title": chunk.get("title", ""),
        "content": chunk.get("content", ""),
        "file_title": chunk.get("file_title", ""),
        "parent_title": chunk.get("parent_title", ""),
        "part": chunk.get("part", 0),
        "has_table": chunk.get("has_table", False),
        "dense": dense_vec,
        "sparse": sparse_dict,
    })

embed_path = RESULT_DIR / "sample_embeddings.json"
embed_path.write_text(
    json.dumps(embeddings_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"\n[落盘] 已写入 {embed_path}（共 {len(embeddings_payload)} 条，含稠密+稀疏向量）")

# ============== 预览 ==============
print("\n=== 前 2 条 chunk 预览 ===")
for i, item in enumerate(embeddings_payload[:2]):
    print(f"\n--- chunk {i + 1} ---")
    print(f"title: {item['title'][:60]}")
    print(f"file_title: {item['file_title']} | parent_title: {item['parent_title'][:40]}")
    print(f"has_table: {item['has_table']} | dense_dim: {len(item['dense'])} | sparse_nnz: {len(item['sparse'])}")
    print(f"sparse 前 3 项: {list(item['sparse'].items())[:3]}")
    print(f"content 前 80 字: {item['content'][:80].replace(chr(10), ' / ')}")